# Imports

In [ ]:
import pathlib
import pandas as pd
import numpy as np
import torch

from hypertrees.models import (
    HyperTreeNetAR,
    HyperTreeAR
)

from experiments.utils import (
    load_experiments_specs,
    create_tsfeatures,
)

# Load Experiment Specifications

In [ ]:
# Load specifications
ablation_run = "A11"
train_type = "global"
dataset = "rossmann"
experiment_specs = load_experiments_specs(
    dataset=dataset,
    train_type=train_type,
)

# Train and Test Data
train_df = experiment_specs["train"]
test_df = experiment_specs["test"]

# Meta Data
meta = experiment_specs["meta"]
features = meta["features"]
fcst_h = meta["fcst_h"]
freq = meta["freq"]

lags = meta["lags"]
max_lag = meta["max_lag"]
n_series = meta["n_series"]
series_ids = meta["series_ids"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed = 123

# Hyper-Parameters
config = experiment_specs["config"]
loss_fn = config["general"]["loss_function"]

dl_params = config["deep_learning"]
htnetar_params = config["hyper_treenet_params"]
htnetar_params_lgb = {k: v for k, v in htnetar_params.items() if k not in ["num_boost_round", "embedding_dimension", "use_random_projection"]}
htar_params = config["hyper_tree_params"]
htnetar_network_params = {k: v for k, v in htnetar_params.items() if k not in ["num_boost_round", "eta", "linear_tree"]}
htnetar_network_params["rp_embed_dim"] = max_lag
htnetar_network_params["hidden_dim"] = dl_params["hidden_size"]
htnetar_network_params["dropout"] = dl_params["dropout"]
htnetar_network_params["learning_rate"] = dl_params["learning_rate"]

# TS-Features

In [ ]:
# Extract time series features
ts_feats_df, ts_feats = create_tsfeatures(
    train=train_df,
    freq=freq
)

# Add features to train and test
train_df = pd.merge(train_df, ts_feats_df, on="series_id", how="left")
test_df = pd.merge(test_df, ts_feats_df, on="series_id", how="left")
features = features + ts_feats

# Ablation:

Instead of using time-varying parameters, we average the parameters across the forecast horizon and use constant AR-parameters for generating forecasts.

### HyperTreeNetAR

In [ ]:
torch.manual_seed(seed)
np.random.seed(seed)

# Initialize
htnet_ar = HyperTreeNetAR(
    p=max_lag,
    freq=freq,
    fcst_h=fcst_h,
    loss_fn=loss_fn,
    device=device
)

# Train
htnet_ar.train(
    lgb_params=htnetar_params_lgb,
    network_params=htnetar_network_params,
    gradient_mode="separate",
    num_iterations=htnetar_params["num_boost_round"],
    train_data=train_df[["series_id", "date", "value"] + features],
    seed=seed,
)

# Average Forecast Parameters
params_fcst_htnet_ar = htnet_ar.forecast(
    test_data=test_df[["series_id", "date"] + features],
    type="parameters"
).drop(columns=["date", "series_id", "model"]).to_numpy().reshape(-1, fcst_h, htnet_ar.p).mean(axis=1, keepdims=True)

# Initialize lags
n_series = train_df["series_id"].nunique()
train_lags = (train_df.groupby(["series_id"], sort=False)
        .apply(lambda x: x["value"][-max(lags):][::-1])
        .reset_index(drop=True)
        .values
        .reshape(n_series, -1)
        )

# Generate forecasts
forecasts_htnet_ar = []
for h in range(fcst_h):
    next_val = np.sum(params_fcst_htnet_ar[:, 0, :] * train_lags, axis=1).reshape(-1, 1)
    forecasts_htnet_ar.append(next_val)
    train_lags = np.concatenate([next_val, train_lags[:, :-1]], axis=1)

htnet_ar_fcst = pd.DataFrame({
    "series_id": test_df["series_id"].to_numpy().flatten(),
    "date": test_df["date"].to_numpy().flatten(),
    "fcst": np.hstack(forecasts_htnet_ar).flatten(),
    "model": "Hyper-TreeNet-AR",
})

# Add actual values
htnet_ar_fcst = pd.merge(
    htnet_ar_fcst,
    test_df[["dataset", "series_id", "date", "value"]],
    on=["series_id", "date"],
    how="inner"
)

### HyperTreeAR

In [ ]:
torch.manual_seed(seed)
np.random.seed(seed)

# Initialize
ht_ar = HyperTreeAR(
    p=max_lag,
    freq=freq,
    fcst_h=fcst_h,
    loss_fn=loss_fn
)

# Train model
ht_ar.train(
    lgb_params=htar_params,
    num_iterations=htar_params["num_boost_round"],
    train_data=train_df[["series_id", "date", "value"] + features],
    seed=seed,
)

# Average Forecast Parameters
params_fcst_htar = ht_ar.forecast(
    test_data=test_df[["series_id", "date"] + features],
    type="parameters"
).drop(columns=["date", "series_id", "model"]).to_numpy().reshape(-1, fcst_h, ht_ar.p).mean(axis=1, keepdims=True)

# Initialize lags
n_series = train_df["series_id"].nunique()
train_lags = (train_df.groupby(["series_id"], sort=False)
        .apply(lambda x: x["value"][-max(lags):][::-1])
        .reset_index(drop=True)
        .values
        .reshape(n_series, -1)
        )

# Generate forecasts
forecasts_ht_ar = []
for h in range(fcst_h):
    next_val = np.sum(params_fcst_htar[:, 0, :] * train_lags, axis=1).reshape(-1, 1)
    forecasts_ht_ar.append(next_val)
    train_lags = np.concatenate([next_val, train_lags[:, :-1]], axis=1)

ht_ar_fcst = pd.DataFrame({
    "series_id": test_df["series_id"].to_numpy().flatten(),
    "date": test_df["date"].to_numpy().flatten(),
    "fcst": np.hstack(forecasts_ht_ar).flatten(),
    "model": "Hyper-Tree-AR",
})

# Add actual values
ht_ar_fcst = pd.merge(
    ht_ar_fcst,
    test_df[["dataset", "series_id", "date", "value"]],
    on=["series_id", "date"],
    how="inner"
)

# Combine forecasts

In [ ]:
fcsts_df = pd.concat(
    [
        htnet_ar_fcst,
        ht_ar_fcst
    ],
    ignore_index=True
)
fcsts_df["ablation"] = ablation_run

# Output

In [ ]:
repo_root = pathlib.Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

result_path = repo_root / "experiments" / "runs" / "results" / "ablation" / dataset.lower()
result_path.mkdir(parents=True, exist_ok=True)

fcsts_df.to_csv(result_path / f"{ablation_run}_fcsts.csv", index=False)